In [ ]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
%env CLEARML_API_ACCESS_KEY=SOTIYZ44P1BVLRCL3HXY09MP35I05O
%env CLEARML_API_SECRET_KEY=qyGfb91AxNXptu67Y-0z1RFAy26g_Kwasldh-GrDnmqR0OuPAKeDYOJcgurrFWTJ058

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=SOTIYZ44P1BVLRCL3HXY09MP35I05O
env: CLEARML_API_SECRET_KEY=qyGfb91AxNXptu67Y-0z1RFAy26g_Kwasldh-GrDnmqR0OuPAKeDYOJcgurrFWTJ058


In [ ]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.9 MB/s eta 0:00:00


In [ ]:
import clearml
print(clearml.__version__)

2.0.2


In [ ]:
from clearml import Task, logger
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
task.close()       # закрываем старый Task
del task

In [ ]:
task_prep = Task.create(
    project_name="Kovalenko_50801",
    task_name="data_preparation",
    task_type=Task.TaskTypes.data_processing
)
logger = task_prep.get_logger()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

fpath = "/content/drive/MyDrive/Drive/transaction.csv"
df_raw = pd.read_csv(fpath)
task_prep.upload_artifact(name='data.raw', artifact_object=fpath)

True

In [ ]:
def aggregate_client_daily_items(df):
     # Приведём названия столбцов к единому стандарту на случай разного написания
    column_mapping = {
        'clientID': 'client',
        'trDte': 'visit_date',
        'itemGroup': 'item_group',
        'itemgroup': 'item_group',
        'ItemGroup': 'item_group'
    }

    # Проверим, какие столбцы есть в df, и переименуем только существующие
    actual_mapping = {k: v for k, v in column_mapping.items() if k in df.columns}
    df_clean = df.rename(columns=actual_mapping)

    # Проверка обязательных колонок
    required = ['client', 'visit_date', 'item', 'item_group', 'quantity', 'amount']
    missing = [col for col in required if col not in df_clean.columns]
    if missing:
        raise ValueError(f"Отсутствуют необходимые столбцы: {missing}")

    # Группировка по (client, visit_date, item, item_group)
    grouped = (
        df_clean
        .groupby(['client', 'visit_date', 'item', 'item_group'], as_index=False)
        .agg({
            'quantity': 'sum',
            'amount': 'sum'
        })
    )

    # Сортировка
    result = grouped.sort_values(by=['client', 'visit_date', 'item']).reset_index(drop=True)

    return result

In [ ]:
visits_df = aggregate_client_daily_items(df_raw)
visits_df

,client,visit_date,item,item_group,quantity,amount
0,client1,22.01.2018,sku10765,Лаки и краски,1,29
1,client1,22.01.2018,sku13695,Стойматериалы,5,1535
2,client1,22.01.2018,sku29083,Лаки и краски,2,310
3,client1,22.01.2018,sku2954,Лаки и краски,1,399
4,client10,05.08.2019,sku1893,Инструменты,1,79
...,...,...,...,...,...,...
1003078,client9999,02.03.2019,sku7708,Комнатные растения и цветы,1,24
1003079,client9999,12.02.2018,sku26016,Стойматериалы,15,7275
1003080,client9999,17.03.2018,sku12516,Инструменты,1,175
1003081,client9999,17.03.2018,sku15869,Инструменты,2,498


In [ ]:
def calculate_client_profile_at_date(visits_df, observation_end_date):
    # Приведение типов
    observation_end_date = pd.to_datetime(observation_end_date)
    visits_df = visits_df.copy()
    visits_df['visit_date'] = pd.to_datetime(visits_df['visit_date'])

    # 1. Фильтрация: только визиты ДО observation_end_date (важно! нет утечки)
    filtered_visits = visits_df[visits_df['visit_date'] < observation_end_date].copy()

    if filtered_visits.empty:
        return pd.DataFrame(columns=[
            'client', 'Recency', 'Frequency', 'Monetary',
            'last_visit_date', 'total_quantity', 'avg_check',
            'total_unique_items', 'avg_items_per_visit',
            'weekend_visits', 'amount_last_visit'
        ])

    # Преобразуем дату в день (удаляем время, если есть)
    filtered_visits['visit_date'] = filtered_visits['visit_date'].dt.date
    observation_date = observation_end_date.date()

    # Группировка по клиенту
    grouped = filtered_visits.groupby('client')

    # Подготовка данных по клиентам
    profiles = []

    for client_id, group in grouped:
        visit_dates = pd.to_datetime(group['visit_date'])  # для вычислений
        last_date = visit_dates.max().date()
        recency = (observation_date - last_date).days

        frequency = group['visit_date'].nunique()  # уникальные даты визитов
        monetary = group['amount'].sum()
        total_quantity = group['quantity'].sum()
        avg_check = monetary / frequency if frequency > 0 else 0
        total_unique_items = group['item'].nunique()
        avg_items_per_visit = total_quantity / frequency

        # Количество визитов в выходные (суббота=5, воскресенье=6)
        visit_weekdays = pd.to_datetime(group['visit_date']).dt.weekday
        weekend_visits = visit_weekdays.isin([5, 6]).sum()

        # Сумма последнего визита
        last_visit_data = group[group['visit_date'] == last_date]
        amount_last_visit = last_visit_data['amount'].sum()

        profiles.append({
            'client': client_id,
            'Recency': recency,
            'Frequency': frequency,
            'Monetary': monetary,
            'last_visit_date': last_date,
            'total_quantity': total_quantity,
            'avg_check': round(avg_check, 2),
            'total_unique_items': total_unique_items,
            'avg_items_per_visit': round(avg_items_per_visit, 2),
            'weekend_visits': weekend_visits,
            'amount_last_visit': amount_last_visit
        })

    # Создаём DataFrame
    result_df = pd.DataFrame(profiles)

    # Сортировка
    result_df = result_df.sort_values(by='client').reset_index(drop=True)

    print(f"✅ Профили рассчитаны: {len(result_df)} клиентов на дату {observation_end_date.date()}")
    return result_df



In [ ]:
# Рассчитываем профиль на 2019-09-01
profile_df = calculate_client_profile_at_date(visits_df, '2019-09-01')

print(profile_df.head())

✅ Профили рассчитаны: 39906 клиентов на дату 2019-09-01
        client  Recency  Frequency  Monetary last_visit_date  total_quantity  \
0      client1      587          1      2273      2018-01-22               9   
1     client10       27          1      4757      2019-08-05               3   
2    client100      116          1      7299      2019-05-08               1   
3   client1000        8         12     31792      2019-08-24             151   
4  client10000      396          1      8495      2018-08-01               5   

   avg_check  total_unique_items  avg_items_per_visit  weekend_visits  \
0    2273.00                   4                 9.00               0   
1    4757.00                   3                 3.00               0   
2    7299.00                   1                 1.00               0   
3    2649.33                  28                12.58              12   
4    8495.00                   5                 5.00               0   

   amount_last_visit  
0

In [ ]:
def mark_events(visits_df, result_start_date, result_end_date):
      # Приведение типов
    result_start = pd.to_datetime(result_start_date)
    result_end = pd.to_datetime(result_end_date)
    visits = visits_df.copy()
    visits['visit_date'] = pd.to_datetime(visits['visit_date'])

    # 1. Получить всех уникальных клиентов
    all_clients = pd.DataFrame({'client': visits['client'].unique()})
    all_clients = all_clients.sort_values('client').reset_index(drop=True)

    # 2. Фильтрация визитов в период [result_start, result_end)
    event_visits = visits[
        (visits['visit_date'] >= result_start) &
        (visits['visit_date'] < result_end)
    ]

    # 3. Определяем клиентов, которые посетили в этот период
    clients_with_visit = event_visits['client'].unique()

    # 4. Добавляем флаг event
    all_clients['event'] = all_clients['client'].isin(clients_with_visit)

    print(f"✅ Разметка событий завершена:")
    print(f"   Всего клиентов: {len(all_clients)}")
    print(f"   Вернулись в период [{result_start.date()} — {result_end.date()}): {all_clients['event'].sum()} ({all_clients['event'].mean():.1%})")

    return all_clients

In [ ]:
# Размечаем событие: [2019-09-01, 2019-10-01)
target_semptember = mark_events(visits_df, '2019-09-01', '2019-10-01')

print(target_semptember)

✅ Разметка событий завершена:
   Всего клиентов: 42746
   Вернулись в период [2019-09-01 — 2019-10-01): 8821 (20.6%)
            client  event
0          client1  False
1         client10  False
2        client100  False
3       client1000   True
4      client10000  False
...            ...    ...
42741   client9995  False
42742   client9996  False
42743   client9997   True
42744   client9998   True
42745   client9999  False

[42746 rows x 2 columns]


In [ ]:
def create_training_sample(profile_df, events_df, visits_df, observation_end_date):
    # Приведение типов
    observation_end_date = pd.to_datetime(observation_end_date)

    # 1. Inner join по 'client'
    sample = profile_df.merge(events_df, on='client', how='inner')
    print(f"✅ Inner join завершён: {len(sample)} клиентов в обеих таблицах")

    # 2. Проверка на пропуски в 'event'
    if sample['event'].isnull().any():
        raise ValueError("Целевая переменная 'event' содержит пропуски!")

    # 3. Восстановим visit_date в profile_df для дальнейших вычислений
    profile_with_date = sample.copy()
    profile_with_date['last_visit_date'] = pd.to_datetime(profile_with_date['last_visit_date'])

    # Подготовка visits_df
    visits = visits_df.copy()
    visits['visit_date'] = pd.to_datetime(visits['visit_date'])

    # Фильтруем визиты ДО observation_end_date
    visits_filtered = visits[visits['visit_date'] < observation_end_date]

    # Добавляем признак: количество уникальных товарных групп
    item_group_agg = visits_filtered.groupby('client')['item_group'].nunique().reset_index()
    item_group_agg.columns = ['client', 'unique_item_groups']

    sample = sample.merge(item_group_agg, on='client', how='left')
    sample['unique_item_groups'] = sample['unique_item_groups'].fillna(0)

    # Убеждаемся, что `amount_last_visit` уже есть (добавлен в профиль)
    if 'amount_last_visit' not in sample.columns:
        raise ValueError("Признак 'amount_last_visit' отсутствует в профиле!")

    # 4. Удаляем служебные колонки с датами
    cols_to_drop = ['last_visit_date']  # можно добавить другие, если есть
    cols_to_drop = [col for col in cols_to_drop if col in sample.columns]
    sample = sample.drop(columns=cols_to_drop)

    # 5. Проверка классов
    event_value_counts = sample['event'].value_counts()
    if len(event_value_counts) < 2:
        print("⚠️ В выборке отсутствует один из классов!")
    else:
        false_count = event_value_counts.get(False, 0)
        true_count = event_value_counts.get(True, 0)
        total = false_count + true_count
        print(f"📊 Распределение классов:")
        print(f"   False: {false_count} ({false_count / total:.1%})")
        print(f"   True:  {true_count} ({true_count / total:.1%})")

    # 6. Финальный список признаков (в нужном порядке)
    feature_columns = [
        'client', 'Recency', 'Frequency', 'Monetary',
        'total_quantity', 'avg_check', 'total_unique_items',
        'avg_items_per_visit', 'weekend_visits',
        'amount_last_visit', 'unique_item_groups',  # ← новый признак
        'event'
    ]

    # Проверка, что все колонки присутствуют
    missing = [col for col in feature_columns if col not in sample.columns]
    if missing:
        raise ValueError(f"Отсутствуют колонки: {missing}")

    result = sample[feature_columns].sort_values(by='client').reset_index(drop=True)

    # Проверка размера
    print(f"✅ Итоговый размер выборки: {len(result)} записей")

    return result

In [ ]:
profile_df = calculate_client_profile_at_date(
    visits_df=visits_df,
    observation_end_date='2019-09-01'
)

events_df = mark_events(
    visits_df=visits_df,
    result_start_date='2019-09-01',
    result_end_date='2019-10-01'
)

training_sample = create_training_sample(
    profile_df=profile_df,
    events_df=events_df,
    visits_df=visits_df,
    observation_end_date='2019-09-01'
)

✅ Профили рассчитаны: 39906 клиентов на дату 2019-09-01
✅ Разметка событий завершена:
   Всего клиентов: 42746
   Вернулись в период [2019-09-01 — 2019-10-01): 8821 (20.6%)
✅ Inner join завершён: 39906 клиентов в обеих таблицах
📊 Распределение классов:
   False: 32375 (81.1%)
   True:  7531 (18.9%)
✅ Итоговый размер выборки: 39906 записей


In [ ]:
logger.report_text(f"training_sample: {training_sample.shape}")

training_sample: (39906, 12)


In [ ]:
profile_test = calculate_client_profile_at_date(
    visits_df=visits_df,
    observation_end_date='2019-10-01'
)

events_test = mark_events(
    visits_df=visits_df,
    result_start_date='2019-10-01',
    result_end_date='2019-11-01'
)

test_data = create_training_sample(
    profile_df=profile_test,
    events_df=events_test,
    visits_df=visits_df,
    observation_end_date='2019-10-01'
)


✅ Профили рассчитаны: 41196 клиентов на дату 2019-10-01
✅ Разметка событий завершена:
   Всего клиентов: 42746
   Вернулись в период [2019-10-01 — 2019-11-01): 9324 (21.8%)
✅ Inner join завершён: 41196 клиентов в обеих таблицах
📊 Распределение классов:
   False: 33422 (81.1%)
   True:  7774 (18.9%)
✅ Итоговый размер выборки: 41196 записей


In [ ]:
logger.report_text(f"test_data: {test_data.shape}")


test_data: (41196, 12)


In [ ]:
TRAIN_PATH = "/content/training_sample.csv"
TEST_PATH  = "/content/test_data.csv"

training_sample.to_csv(TRAIN_PATH, index=False)
test_data.to_csv(TEST_PATH, index=False)

print("Files saved:", TRAIN_PATH, TEST_PATH)

Files saved: /content/training_sample.csv /content/test_data.csv


In [ ]:
import os
os.listdir("/content")

['.config',
 'training_sample.csv',
 'binning_process.pkl',
 'xgboost_model.pkl',
 'logreg_all_features.pkl',
 'train_woe.csv',
 'test_woe.csv',
 'drive',
 'test_data.csv',
 'sample_data']

In [ ]:
from clearml import Dataset

dataset = Dataset.create(
    dataset_project="Kovalenko_50801",
    dataset_name="customer_features"
)

dataset.add_files(TRAIN_PATH)
dataset.add_files(TEST_PATH)

dataset.set_description(
    "Customer features dataset generated from visits_df. "
    "Includes training_sample and test_data."
)

dataset.upload()
dataset.finalize()

ClearML results page: https://app.clear.ml/projects/89266d9dc7e54c32b900caf5f8aefb80/experiments/5bbb610cfab54a7dbfb9fac7407ef487/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/89266d9dc7e54c32b900caf5f8aefb80/experiments/5bbb610cfab54a7dbfb9fac7407ef487


In [ ]:
task_prep.close()  # закрываем старый task


In [ ]:
from clearml import Task, Dataset
import warnings
warnings.filterwarnings("ignore")

task_woe_iv = Task.create(
    project_name="Kovalenko_50801",
    task_name="woe_iv_optbinning",
    task_type=Task.TaskTypes.data_processing
)

logger = task_woe_iv.get_logger()

In [ ]:
import pandas as pd
import os

dataset = Dataset.get(
    dataset_project="Kovalenko_50801",
    dataset_name="customer_features",
    dataset_version="1.0.0"
)

data_path = dataset.get_local_copy()

train = pd.read_csv(os.path.join(data_path, "training_sample.csv"))
test  = pd.read_csv(os.path.join(data_path, "test_data.csv"))

logger.report_text(f"Train shape: {train.shape}")
logger.report_text(f"Test shape: {test.shape}")


Train shape: (39906, 12)
Test shape: (41196, 12)


In [ ]:
candidate_targets = ["target","TARGET","y","Y","label","Label","event","Event","response"]
target_col = next((c for c in candidate_targets if c in train.columns), train.columns[-1])

logger.report_text(f"Target column: {target_col}")

train[target_col] = train[target_col].astype(int)
if target_col in test.columns:
    test[target_col] = test[target_col].astype(int)

Target column: event


In [ ]:
!pip install optbinning

In [ ]:
import numpy as np
from optbinning import OptimalBinning

MAX_N_BINS = 5
MIN_BIN_SIZE = 0.05

excluded = {target_col}
excluded |= {c for c in ["id","ID","Id"] if c in train.columns}
features = [c for c in train.columns if c not in excluded]

y = train[target_col]
total_events = y.sum()
total_non_events = len(y) - total_events

binning_objects = {}
iv_scores = {}
woe_tables = {}


In [ ]:
def detect_type(series):
    return "numerical" if pd.api.types.is_numeric_dtype(series) and series.nunique() > 10 else "categorical"


In [ ]:
for feat in features:
    try:
        optb = OptimalBinning(
            name=feat,
            dtype=detect_type(train[feat]),
            max_n_bins=MAX_N_BINS,
            min_bin_size=MIN_BIN_SIZE
        )
        optb.fit(train[feat].values, y.values)

        bins = optb.transform(train[feat].values, metric="bins")
        dfb = pd.DataFrame({"bin": bins, "y": y.values})

        grp = dfb.groupby("bin")["y"].agg(['count','sum']).reset_index()
        grp.rename(columns={'sum':'events'}, inplace=True)
        grp['non_events'] = grp['count'] - grp['events']

        grp['distr_events'] = grp['events'] / (total_events + 1e-10)
        grp['distr_non_events'] = grp['non_events'] / (total_non_events + 1e-10)

        grp['woe'] = np.log(grp['distr_events'] / grp['distr_non_events'])
        grp['iv'] = (grp['distr_events'] - grp['distr_non_events']) * grp['woe']

        iv_scores[feat] = grp['iv'].sum()
        binning_objects[feat] = optb
        woe_tables[feat] = grp

        logger.report_scalar("IV", feat, iv_scores[feat])

    except Exception as e:
        logger.report_text(f"FAILED {feat}: {e}")


FAILED client: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED Recency: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED Frequency: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED Monetary: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED total_quantity: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED avg_check: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED total_unique_items: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED avg_items_per_visit: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED weekend_visits: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED amount_last_visit: Logger.report_scalar() missing 1 required positional argument: 'iteration'
FAILED unique_item_groups: Logger.repor

In [ ]:
train_woe = train.copy()
test_woe  = test.copy()

for feat, optb in binning_objects.items():
    train_woe[feat] = optb.transform(train[feat].values, metric="woe")
    test_woe[feat]  = optb.transform(test[feat].values, metric="woe")

In [ ]:
import pickle

train_woe.to_csv("train_woe.csv", index=False)
test_woe.to_csv("test_woe.csv", index=False)

with open("binning_process.pkl", "wb") as f:
    pickle.dump({
        "binning_objects": binning_objects,
        "iv_scores": iv_scores,
        "woe_tables": woe_tables,
        "features": features,
        "target_col": target_col
    }, f)

task_woe_iv.upload_artifact("train_woe", "train_woe.csv")
task_woe_iv.upload_artifact("test_woe", "test_woe.csv")
task_woe_iv.upload_artifact("binning_process", "binning_process.pkl")

                                            0% | 0.00/7.72 MB [00:00<?, ?MB/s]: 
                                            0% | 0.00/7.97 MB [00:00<?, ?MB/s]: 

True

In [ ]:
from clearml import Task, Dataset
# Получить ID предыдущего датасета
parent_dataset = Dataset.get(
    dataset_project="Kovalenko_50801",
    dataset_name="customer_features",
    only_completed=True
)

woe_dataset = Dataset.create(
    dataset_name="customer_features_woe",
    dataset_project="Kovalenko_50801",
    parent_datasets=[parent_dataset]  # ← ВАЖНО: связь версий
)


ClearML results page: https://app.clear.ml/projects/d618192bb24947a194226c06f48d9b3a/experiments/cf12659c91514a74a3830131e671315a/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/d618192bb24947a194226c06f48d9b3a/experiments/cf12659c91514a74a3830131e671315a


In [ ]:
woe_dataset = Dataset.create(
    dataset_name="customer_features_woe",
    dataset_project="Kovalenko_50801"
)

woe_dataset.add_files("train_woe.csv")
woe_dataset.add_files("test_woe.csv")
woe_dataset.upload()
woe_dataset.finalize()

logger.report_text("WoE Dataset created successfully")

ClearML results page: https://app.clear.ml/projects/d618192bb24947a194226c06f48d9b3a/experiments/8d385355b3614b6ea1e7ac77ff9ee4ab/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/d618192bb24947a194226c06f48d9b3a/experiments/8d385355b3614b6ea1e7ac77ff9ee4ab
Uploading dataset changes (2 files compressed to 8.02 MiB) to https://files.clear.ml


█████████████████████████████████ 100% | 8.02/8.02 MB [00:00<00:00, 36.82MB/s]: 


File compression and upload completed: total size 8.02 MiB, 1 chunk(s) stored (average size 8.02 MiB)
WoE Dataset created successfully


In [ ]:
# В конце Task "woe_iv_optbinning", перед task.close()
logger.report_text("Data preparation COMPLETE")
logger.report_text(f"Created datasets: customer_features v1.0, customer_features_woe v2.0")
logger.report_text(f"WoE transformation applied to {len(features)} features")

task_woe_iv.close()

Data preparation COMPLETE
Created datasets: customer_features v1.0, customer_features_woe v2.0
WoE transformation applied to 11 features


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from clearml import Task, Dataset

In [ ]:
# ОДИН проект для всех задач
task_xgb = Task.create(
    project_name="Kovalenko_50801",  # 👈 правильный проект
    task_name="train_xgboost_v1",          # уникальное имя Task
    task_type=Task.TaskTypes.training
)

logger = task.get_logger()

In [ ]:
dataset = Dataset.get(
    dataset_project="Kovalenko_50801",
    dataset_name="customer_features",
    dataset_version="1.0.0"   # ← конкретная версия
)

data_path = dataset.get_local_copy()
df = pd.read_csv(f"{data_path}/training_sample.csv")

logger.report_text(f"Dataset loaded: {df.shape}")

Dataset loaded: (39906, 12)


In [ ]:
target_col = "event"

X = df.drop(columns=[target_col, "client"])
y = df[target_col]

# только числовые признаки
X = X.select_dtypes(include=[np.number])


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
xgb_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "eval_metric": "auc",
    "use_label_encoder": False
}

# Фиксация гиперпараметров в ClearML
task.connect(xgb_params)

{'n_estimators': 200,
 'max_depth': 4,
 'learning_rate': 0.1,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'random_state': 42,
 'eval_metric': 'auc',
 'use_label_encoder': False}

In [ ]:
model = xgb.XGBClassifier(**xgb_params)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_val_proba = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_val_proba)

logger.report_single_value(
    name="ROC_AUC",
    value=roc_auc
)

logger.report_text(f"Validation ROC-AUC: {roc_auc:.4f}")

Validation ROC-AUC: 0.8074


In [ ]:
model_path = "xgboost_model.pkl"
joblib.dump(model, model_path)

task.upload_artifact(
    name="xgboost_model",
    artifact_object=model_path
)

True

In [ ]:
task.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from clearml import Task

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from clearml import Task, Dataset

# 1️⃣ Создание Task
task_logreg = Task.create(
    project_name="Kovalenko_50801",
    task_name="logreg_all_features",
    task_type=Task.TaskTypes.training
)
logger = task_logreg.get_logger()

# 2️⃣ Загрузка WoE датасета из ClearML
woe_dataset = Dataset.get(
    dataset_project="Kovalenko_50801",
    dataset_name="customer_features_woe",
    only_completed=True
)
data_path = woe_dataset.get_local_copy()

train_woe = pd.read_csv(os.path.join(data_path, "train_woe.csv"))
test_woe = pd.read_csv(os.path.join(data_path, "test_woe.csv"))

logger.report_text(f"Train WoE shape: {train_woe.shape}")
logger.report_text(f"Test WoE shape: {test_woe.shape}")

# 3️⃣ Гиперпараметры
params = {
    "model_type": "LogisticRegression",
    "max_iter": 1000,
    "C": 1.0,
    "penalty": "l2",
    "solver": "lbfgs"
}
task_logreg.connect(params)

# 4️⃣ Подготовка данных
target_col = "event"
if target_col not in train_woe.columns:
    target_col = next((c for c in ["event", "target", "y"] if c in train_woe.columns), None)
    if target_col is None:
        raise ValueError("Target column not found!")

X_train = train_woe.drop(columns=[target_col]).select_dtypes(include=[np.number])
y_train = train_woe[target_col]

X_test = test_woe.drop(columns=[target_col]).select_dtypes(include=[np.number])
y_test = test_woe[target_col] if target_col in test_woe.columns else None

logger.report_text(f"Features used: {list(X_train.columns)}")

# 5️⃣ Обучение модели
model = LogisticRegression(
    max_iter=params["max_iter"],
    C=params["C"],
    penalty=params["penalty"],
    solver=params["solver"],
    random_state=42
)
model.fit(X_train, y_train)

# 6️⃣ Предсказания и метрики
y_train_pred_proba = model.predict_proba(X_train)[:, 1]
auc_train = roc_auc_score(y_train, y_train_pred_proba)
logger.report_scalar(title="ROC-AUC", series="train", value=auc_train, iteration=0)

if y_test is not None:
    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, y_test_pred_proba)
    logger.report_scalar(title="ROC-AUC", series="test", value=auc_test, iteration=0)

# 7️⃣ Сохраняем модель
model_path = "logreg_all_features.pkl"
joblib.dump(model, model_path)
task_logreg.upload_artifact(name="logreg_model.pkl", artifact_object=model_path)

# 8️⃣ ROC-кривая — отправляем в ClearML
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC={auc_train:.2f})", color='blue')

if y_test is not None:
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)
    plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC={auc_test:.2f})", color='red')

plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve — Logistic Regression (All features)")
plt.legend()
plt.grid(True)
plt.tight_layout()

# Вместо plt.show() используем ClearML logger
logger.report_matplotlib_figure(
    title="ROC Curve",
    series="train_vs_test",
    figure=plt.gcf(),
    iteration=0
)
plt.close()  # закрываем фигуру, чтобы не отображалась дважды в Colab

# 9️⃣ Завершение Task
task_logreg.close()

                                            0% | 0.00/8.02 MB [00:00<?, ?MB/s]: /usr/local/lib/python3.12/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
████████████████████████████████ 100% | 8.02/8.02 MB [00:00<00:00, 348.88MB/s]: 


Train WoE shape: (39906, 12)
Test WoE shape: (41196, 12)
Features used: ['client', 'Recency', 'Frequency', 'Monetary', 'total_quantity', 'avg_check', 'total_unique_items', 'avg_items_per_visit', 'weekend_visits', 'amount_last_visit', 'unique_item_groups']


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from clearml import Task

# 0️⃣ Инициализация ClearML Task
task_logreg = Task.create(
    project_name="Kovalenko_50801",
    task_name="logreg_numeric_features",
    task_type=Task.TaskTypes.training
)
logger = task_logreg.get_logger()

# 1️⃣ Определяем X и y
target = target_col  # таргет

X_train = train_woe.drop(columns=[target])
y_train = train_woe[target]

X_test = test_woe.drop(columns=[target])
y_test = test_woe[target] if target in test_woe.columns else None

# Выбираем только числовые признаки (WoE уже числовые)
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test[X_train.columns]  # согласуем колонки

logger.report_text(f"Features used: {list(X_train.columns)}")

# 2️⃣ Обучаем логистическую регрессию
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

# 3️⃣ Предсказания (вероятности)
y_train_pred_proba = logreg.predict_proba(X_train)[:, 1]
y_test_pred_proba  = logreg.predict_proba(X_test)[:, 1]

# 4️⃣ ROC-AUC и логирование
auc_train = roc_auc_score(y_train, y_train_pred_proba)
logger.report_scalar(title="ROC-AUC", series="train", value=auc_train, iteration=0)

if y_test is not None:
    auc_test = roc_auc_score(y_test, y_test_pred_proba)
    logger.report_scalar(title="ROC-AUC", series="test", value=auc_test, iteration=0)

# 5️⃣ ROC-кривые — логирование через ClearML
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC={auc_train:.2f})", color='blue')

if y_test is not None:
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)
    plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC={auc_test:.2f})", color='red')

plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve — Logistic Regression (Numeric features)")
plt.legend()
plt.grid(True)
plt.tight_layout()

# Отправляем график в ClearML
logger.report_matplotlib_figure(
    title="ROC Curve",
    series="train_vs_test",
    figure=plt.gcf(),
    iteration=0
)
plt.close()  # закрываем фигуру, чтобы не дублировалась в Colab

# 6️⃣ Завершение Task
task_logreg.close()

Features used: ['client', 'Recency', 'Frequency', 'Monetary', 'total_quantity', 'avg_check', 'total_unique_items', 'avg_items_per_visit', 'weekend_visits', 'amount_last_visit', 'unique_item_groups']


In [ ]:
# 0️⃣ Инициализация ClearML Task
task_logreg = Task.create(
    project_name="Kovalenko_50801",
    task_name="trivial_vs_multi_logreg",
    task_type=Task.TaskTypes.training
)
logger = task_logreg.get_logger()

# -----------------------------
# 1️⃣ Тривиальная модель (Recency)
# -----------------------------
X_trivial_train = train_woe[["Recency"]]
y_train = train_woe[target_col]
X_trivial_test = test_woe[["Recency"]]
y_test = test_woe[target_col] if target_col in test_woe.columns else None

logreg_trivial = LogisticRegression()
logreg_trivial.fit(X_trivial_train, y_train)

y_trivial_train_proba = logreg_trivial.predict_proba(X_trivial_train)[:, 1]
y_trivial_test_proba  = logreg_trivial.predict_proba(X_trivial_test)[:, 1]

auc_trivial_train = roc_auc_score(y_train, y_trivial_train_proba)
logger.report_scalar("ROC-AUC", "Trivial Train", auc_trivial_train, iteration=0)
auc_trivial_test  = roc_auc_score(y_test, y_trivial_test_proba) if y_test is not None else None
if auc_trivial_test is not None:
    logger.report_scalar("ROC-AUC", "Trivial Test", auc_trivial_test, iteration=0)

fpr_trivial_train, tpr_trivial_train, _ = roc_curve(y_train, y_trivial_train_proba)
fpr_trivial_test, tpr_trivial_test, _ = roc_curve(y_test, y_trivial_test_proba) if y_test is not None else (None, None, None)

# -----------------------------
# 2️⃣ Многофакторная модель (все признаки)
# -----------------------------
X_multi_train = train_woe.drop(columns=[target_col]).select_dtypes(include=[np.number])
X_multi_test  = test_woe[X_multi_train.columns]

logreg_multi = LogisticRegression(max_iter=1000)
logreg_multi.fit(X_multi_train, y_train)

y_multi_train_proba = logreg_multi.predict_proba(X_multi_train)[:, 1]
y_multi_test_proba  = logreg_multi.predict_proba(X_multi_test)[:, 1]

auc_multi_train = roc_auc_score(y_train, y_multi_train_proba)
logger.report_scalar("ROC-AUC", "Multi Train", auc_multi_train, iteration=0)
auc_multi_test  = roc_auc_score(y_test, y_multi_test_proba) if y_test is not None else None
if auc_multi_test is not None:
    logger.report_scalar("ROC-AUC", "Multi Test", auc_multi_test, iteration=0)

fpr_multi_train, tpr_multi_train, _ = roc_curve(y_train, y_multi_train_proba)
fpr_multi_test, tpr_multi_test, _ = roc_curve(y_test, y_multi_test_proba) if y_test is not None else (None, None, None)

# -----------------------------
# 3️⃣ Построение объединенного графика
# -----------------------------
plt.figure(figsize=(8, 6))

# Тривиальная модель
plt.plot(fpr_trivial_train, tpr_trivial_train, label=f"Trivial Train (AUC={auc_trivial_train:.2f})", color='blue', linestyle='--')
if y_test is not None:
    plt.plot(fpr_trivial_test, tpr_trivial_test, label=f"Trivial Test (AUC={auc_trivial_test:.2f})", color='blue')

# Многофакторная модель
plt.plot(fpr_multi_train, tpr_multi_train, label=f"Multi Train (AUC={auc_multi_train:.2f})", color='red', linestyle='--')
if y_test is not None:
    plt.plot(fpr_multi_test, tpr_multi_test, label=f"Multi Test (AUC={auc_multi_test:.2f})", color='red')

# Случайная модель
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — Trivial vs Multi-factor Logistic Regression")
plt.legend()
plt.grid(True)
plt.tight_layout()

# Отправляем график в ClearML
logger.report_matplotlib_figure(
    title="ROC Curves",
    series="Trivial_vs_Multi",
    figure=plt.gcf(),
    iteration=0
)
plt.close()  # закрываем фигуру, чтобы не отображалась в Colab

# -----------------------------
# 4️⃣ Завершение Task
# -----------------------------
task_logreg.close()

In [ ]:
# 0️⃣ Инициализация ClearML Task
task_forecast = Task.create(
    project_name="Kovalenko_50801",
    task_name="final_forecast",
    task_type=Task.TaskTypes.inference
)
logger = task_forecast.get_logger()

print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ ЭТАП: ПРОГНОЗ НА СЛЕДУЮЩИЙ МЕСЯЦ")
print("=" * 70)

# 1️⃣ Дата построения профиля и прогнозного периода
observation_date_for_forecast = '2019-10-01'
forecast_start_date = '2019-11-01'
forecast_end_date = '2019-12-01'

logger.report_text(f"Дата построения профиля: {observation_date_for_forecast}")
logger.report_text(f"Прогнозный период: с {forecast_start_date} по {forecast_end_date}")

# 2️⃣ Подготовка данных для прогноза
X_forecast = test_woe.copy()
if target_col in X_forecast.columns:
    X_forecast = X_forecast.drop(columns=[target_col])

X_forecast_numeric = X_forecast.select_dtypes(include=[np.number])
X_forecast_aligned = X_forecast_numeric.reindex(columns=X_train.columns, fill_value=0)

logger.report_text(f"Признаков для прогноза: {X_forecast_aligned.shape[1]}")
logger.report_text(f"Клиентов для прогноза: {X_forecast_aligned.shape[0]}")

# 3️⃣ Загружаем лучшую модель
try:
    with open("best_model.pkl", "rb") as f:
        best_model_loaded = pickle.load(f)
    logger.report_text("✅ Лучшая модель успешно загружена")
except FileNotFoundError:
    logger.report_text("⚠️ Лучшая модель не найдена, используем многофакторную логрегрессию")
    best_model_loaded = logreg_multi

logger.report_text(f"Тип используемой модели: {type(best_model_loaded).__name__}")

# 4️⃣ Выполняем прогноз
forecast_probabilities = best_model_loaded.predict_proba(X_forecast_aligned)
return_probabilities = forecast_probabilities[:, 1] if forecast_probabilities.shape[1] == 2 else forecast_probabilities
forecast_classes = best_model_loaded.predict(X_forecast_aligned)

# 5️⃣ Создаем итоговый DataFrame
forecast_results = pd.DataFrame({
    'client': test_woe.index if 'id' not in test_woe.columns else test_woe['id'],
    'probability_return': return_probabilities,
    'predicted_class': forecast_classes
})

for col in ['Recency', 'Frequency', 'Monetary']:
    forecast_results[col] = X_forecast_aligned[col] if col in X_forecast_aligned.columns else 0

# 6️⃣ Сортировка по вероятности возврата
forecast_results_sorted = forecast_results.sort_values('probability_return', ascending=False).reset_index(drop=True)

# 7️⃣ Статистика прогноза
mean_prob = forecast_results_sorted['probability_return'].mean()
median_prob = forecast_results_sorted['probability_return'].median()
std_prob = forecast_results_sorted['probability_return'].std()

logger.report_text("\n📈 СТАТИСТИКА ПРОГНОЗА:")

# Обязательный аргумент iteration добавлен
logger.report_scalar("Forecast stats", "Mean probability", mean_prob, iteration=0)
logger.report_scalar("Forecast stats", "Median probability", median_prob, iteration=0)
logger.report_scalar("Forecast stats", "Std deviation", std_prob, iteration=0)

class_counts = forecast_results_sorted['predicted_class'].value_counts()
for cls, count in class_counts.items():
    logger.report_text(f"Прогнозируемый класс {cls}: {count} клиентов ({count / len(forecast_results_sorted) * 100:.1f}%)")


# 8️⃣ Топ-20 клиентов
logger.report_text("\n🏆 ТОП-20 КЛИЕНТОВ С НАИБОЛЬШЕЙ ВЕРОЯТНОСТЬЮ ВОЗВРАТА:")
logger.report_table(title="Top 20 clients", series="forecast", table_plot=forecast_results_sorted.head(20))

# 9️⃣ Визуализация распределения вероятностей
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.hist(forecast_results_sorted['probability_return'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Вероятность возврата')
plt.ylabel('Количество клиентов')
plt.title('Распределение вероятностей')
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.boxplot(forecast_results_sorted['probability_return'], vert=False)
plt.xlabel('Вероятность возврата')
plt.title('Box plot вероятностей')
plt.grid(True, alpha=0.3)

plt.tight_layout()

# Логируем график в ClearML
logger.report_matplotlib_figure(
    title="Forecast Probabilities Distribution",
    series="forecast_distribution",
    figure=plt.gcf(),
    iteration=0
)
plt.close()

# 10️⃣ Сохраняем прогноз
os.makedirs('results', exist_ok=True)
forecast_results_sorted.to_csv('results/forecast.csv', index=False, encoding='utf-8-sig')
logger.report_text("💾 Прогноз сохранен в 'results/forecast.csv'")

# 11️⃣ Проверка сохраненного файла
saved_forecast = pd.read_csv('results/forecast.csv', encoding='utf-8-sig')
logger.report_text("\nПервые 5 строк сохраненного файла:")
logger.report_table(title="Saved forecast preview", series="forecast_preview", table_plot=saved_forecast.head())

# 12️⃣ Завершение Task
task_forecast.close()


ФИНАЛЬНЫЙ ЭТАП: ПРОГНОЗ НА СЛЕДУЮЩИЙ МЕСЯЦ
Дата построения профиля: 2019-10-01
Прогнозный период: с 2019-11-01 по 2019-12-01
Признаков для прогноза: 11
Клиентов для прогноза: 41196
⚠️ Лучшая модель не найдена, используем многофакторную логрегрессию
Тип используемой модели: LogisticRegression

📈 СТАТИСТИКА ПРОГНОЗА:
Прогнозируемый класс 0: 37953 клиентов (92.1%)
Прогнозируемый класс 1: 3243 клиентов (7.9%)

🏆 ТОП-20 КЛИЕНТОВ С НАИБОЛЬШЕЙ ВЕРОЯТНОСТЬЮ ВОЗВРАТА:
💾 Прогноз сохранен в 'results/forecast.csv'

Первые 5 строк сохраненного файла:


In [ ]:
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from clearml import Task

# 0️⃣ Инициализация ClearML Task
task_model = Task.create(
    project_name="Kovalenko_50801",
    task_name="model_selection_forecast",
    task_type=Task.TaskTypes.training
)
logger = task_model.get_logger()

# -----------------------------
# 1️⃣ Подготовка данных
# -----------------------------
target = target_col
X = train_woe.drop(columns=[target]).select_dtypes(include=[np.number])
y = train_woe[target]

# Разделяем на тренировку и валидацию
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
logger.report_text(f"Количество признаков: {X_train.shape[1]}")
logger.report_text(f"Тренировочных объектов: {X_train.shape[0]}, валидационных объектов: {X_val.shape[0]}")

# -----------------------------
# 2️⃣ Обучение XGBoost
# -----------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)
xgb_model.fit(X_train, y_train)
y_val_pred_xgb = xgb_model.predict_proba(X_val)[:, 1]
auc_xgb = roc_auc_score(y_val, y_val_pred_xgb)
logger.report_scalar("ROC-AUC", "XGBoost (val)", auc_xgb, iteration=0)

# -----------------------------
# 3️⃣ Обучение многофакторной логрегрессии
# -----------------------------
logreg_model = LogisticRegression(max_iter=1000)
logreg_model.fit(X_train, y_train)
y_val_pred_logreg = logreg_model.predict_proba(X_val)[:, 1]
auc_logreg = roc_auc_score(y_val, y_val_pred_logreg)
logger.report_scalar("ROC-AUC", "Logistic Regression (val)", auc_logreg, iteration=0)

# -----------------------------
# 4️⃣ Выбор лучшей модели
# -----------------------------
if auc_xgb >= auc_logreg:
    best_model = xgb_model
    best_model_name = "XGBoost"
    best_auc = auc_xgb
else:
    best_model = logreg_model
    best_model_name = "Logistic Regression"
    best_auc = auc_logreg

logger.report_text(f"\n✅ Лучшая модель: {best_model_name} (ROC-AUC={best_auc:.4f})")

# -----------------------------
# 5️⃣ Подготовка данных для прогноза
# -----------------------------
X_forecast = test_woe.copy()
if target_col in X_forecast.columns:
    X_forecast = X_forecast.drop(columns=[target_col])
X_forecast_numeric = X_forecast.select_dtypes(include=[np.number])
X_forecast_aligned = X_forecast_numeric.reindex(columns=X_train.columns, fill_value=0)

# -----------------------------
# 6️⃣ Прогноз
# -----------------------------
forecast_probabilities = best_model.predict_proba(X_forecast_aligned)
return_probabilities = forecast_probabilities[:, 1] if forecast_probabilities.shape[1] == 2 else forecast_probabilities
forecast_classes = best_model.predict(X_forecast_aligned)

# -----------------------------
# 7️⃣ Итоговый DataFrame с прогнозами
# -----------------------------
forecast_results = pd.DataFrame({
    'client': test_woe.index if 'id' not in test_woe.columns else test_woe['id'],
    'probability_return': return_probabilities,
    'predicted_class': forecast_classes
})

for col in ['Recency', 'Frequency', 'Monetary']:
    forecast_results[col] = X_forecast_aligned[col] if col in X_forecast_aligned.columns else 0

forecast_results_sorted = forecast_results.sort_values('probability_return', ascending=False).reset_index(drop=True)

# -----------------------------
# 8️⃣ ROC-кривая для лучшей модели
# -----------------------------
fpr, tpr, _ = roc_curve(y_val, best_model.predict_proba(X_val)[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"{best_model_name} ROC (AUC={best_auc:.2f})", color='blue')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC curve — {best_model_name}")
plt.legend()
plt.grid(True)
plt.tight_layout()
logger.report_matplotlib_figure(
    title=f"{best_model_name} ROC Curve",
    series="ROC",
    figure=plt.gcf(),
    iteration=0
)
plt.close()

# -----------------------------
# 9️⃣ Сохранение и логирование прогноза
# -----------------------------
os.makedirs('results', exist_ok=True)
forecast_results_sorted.to_csv('results/forecast.csv', index=False, encoding='utf-8-sig')
logger.report_text(f"💾 Прогноз сохранен в 'results/forecast.csv'")
logger.report_text(f"Всего прогнозируемых клиентов: {len(forecast_results_sorted)}")
logger.report_text(f"Лучшая модель: {best_model_name} (ROC-AUC={best_auc:.4f})")
logger.report_table(title="Top 5 forecast", series="forecast_preview", table_plot=forecast_results_sorted.head())

# -----------------------------
# 🔟 Завершение Task
# -----------------------------
task_model.close()


Количество признаков: 11
Тренировочных объектов: 31924, валидационных объектов: 7982


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning:

[15:43:50] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.





✅ Лучшая модель: Logistic Regression (ROC-AUC=0.8086)
💾 Прогноз сохранен в 'results/forecast.csv'
Всего прогнозируемых клиентов: 41196
Лучшая модель: Logistic Regression (ROC-AUC=0.8086)
